# 003 — Cascaded Inference (Multimodal v2)

問診四題 + 音訊 → 516 維特徵 → 兩階段分類

```
音訊 → HeAR → 512維 ┐
問診四題             ├→ Stage 1: covid vs non-covid
  age               │      ├─ covid  → 輸出信心分數
  gender            │      └─ non-covid
  respiratory       │             ↓
  fever             │       Stage 2: healthy vs symptomatic
                    ┘             ├─ healthy     → 輸出信心分數
                                  └─ symptomatic → 輸出三類信心分數
```

## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
import librosa
import tensorflow as tf
from huggingface_hub import from_pretrained_keras
from huggingface_hub.utils import HfFolder

c:\Users\aint\anaconda3\envs\base1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 載入模型

In [8]:
print('載入 HeAR...')
hear_model = from_pretrained_keras('google/hear')
serving_fn  = hear_model.signatures['serving_default']
print('HeAR 載入完成')

ckpt_s1 = joblib.load('checkpoints/stage1_covid_vs_noncovid.pkl')
ckpt_s2 = joblib.load('checkpoints/stage2_healthy_vs_symptomatic.pkl')

clf_s1, scaler_s1, encoder_s1 = ckpt_s1['clf'], ckpt_s1['scaler'], ckpt_s1['encoder']
clf_s2, scaler_s2, encoder_s2 = ckpt_s2['clf'], ckpt_s2['scaler'], ckpt_s2['encoder']
FEATURES = ckpt_s1['features']  # 516 維

print('Stage 1 classes:', encoder_s1.classes_)
print('Stage 2 classes:', encoder_s2.classes_)

載入 HeAR...


Fetching 24 files: 100%|██████████| 24/24 [00:00<?, ?it/s]

HeAR 載入完成
Stage 1 classes: ['covid' 'non-covid']
Stage 2 classes: ['healthy' 'symptomatic']


## 選擇音訊檔案

In [13]:
COUGHVID_ROOT = "C:/Users/aint/Desktop/RNN/Final_project/coughvid_wav"
AUG_ROOT      = 'data/augmented_wav'
TEST_CSV      = 'data/prepared_test_split_hear.csv'

df_test = pd.read_csv(TEST_CSV)
sample  = df_test.sample(1).iloc[0]
sample = df_test.sample(1, random_state=42).iloc[0]

filename = sample['filename']

# 根據檔名決定路徑
if filename.startswith('aug_'):
    fn_wav = os.path.join(AUG_ROOT, filename)
else:
    fn_wav = os.path.join(COUGHVID_ROOT, filename)
print(f'檔案:     {filename}')
print(f'真實標籤: {sample["label"]}')
print(f'路徑:     {fn_wav}')

檔案:     71c1c347-8d5d-434b-ab15-3f0a73d2caf7.wav
真實標籤: symptomatic
路徑:     C:/Users/aint/Desktop/RNN/Final_project/coughvid_wav\71c1c347-8d5d-434b-ab15-3f0a73d2caf7.wav


## 問診四題

實際整合 Llama3 時，這裡的值由問診結果填入。
目前用 test set 的資料模擬。

In [14]:
# 從 test set 取得對應的臨床資訊（模擬問診結果）
age                   = float(sample.get('age', 35))
gender_encoded        = int(sample.get('gender_encoded', -1))
respiratory_condition = int(sample.get('respiratory_condition', 0))
fever_muscle_pain     = int(sample.get('fever_muscle_pain', 0))

gender_display = {0: '男', 1: '女', 2: '其他', -1: '未知'}
print(f'問診結果:')
print(f'  年齡:           {age}')
print(f'  性別:           {gender_display.get(gender_encoded, "未知")}')
print(f'  慢性呼吸道疾病: {"是" if respiratory_condition else "否"}')
print(f'  發燒或肌肉痠痛: {"是" if fever_muscle_pain else "否"}')

問診結果:
  年齡:           29.0
  性別:           男
  慢性呼吸道疾病: 是
  發燒或肌肉痠痛: 是


## HeAR Embedding 提取

In [15]:
SAMPLE_RATE  = 16000
CLIP_SAMPLES = SAMPLE_RATE * 2

def get_embedding(file_path):
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
    if len(audio) < CLIP_SAMPLES:
        audio = np.pad(audio, (0, CLIP_SAMPLES - len(audio)))
    n_clips = len(audio) // CLIP_SAMPLES
    clips   = np.array([audio[i*CLIP_SAMPLES:(i+1)*CLIP_SAMPLES]
                        for i in range(n_clips)], dtype=np.float32)
    return serving_fn(x=tf.constant(clips))['output_0'].numpy().mean(axis=0)

emb = get_embedding(fn_wav)

# 組合 516 維特徵
X_input = np.concatenate([
    emb,
    [age, gender_encoded, respiratory_condition, fever_muscle_pain]
]).reshape(1, -1).astype(np.float32)

print(f'X_input.shape: {X_input.shape}')

X_input.shape: (1, 516)


## Cascaded Inference

In [16]:
CONFIDENCE_THRESHOLD = 0.60  # 信心門檻，低於此值視為「不確定」

# ── Stage 1: covid vs non-covid ──
X_s1    = scaler_s1.transform(X_input)
prob_s1 = clf_s1.predict_proba(X_s1)[0]
pred_s1 = encoder_s1.classes_[np.argmax(prob_s1)]

covid_idx    = list(encoder_s1.classes_).index('covid')
noncovid_idx = list(encoder_s1.classes_).index('non-covid')

print('=' * 45)
print('Stage 1: COVID vs Non-COVID')
print(f'  covid:     {prob_s1[covid_idx]*100:.1f}%')
print(f'  non-covid: {prob_s1[noncovid_idx]*100:.1f}%')
print(f'  → 判定: {pred_s1}')
print()

if pred_s1 == 'covid':
    conf = prob_s1[covid_idx]
    print('=' * 45)
    if conf >= CONFIDENCE_THRESHOLD:
        print(f'最終結果: COVID-19')
        print(f'  信心分數: {conf*100:.1f}%')
    else:
        print(f'最終結果: 疑似 COVID-19（信心不足）')
        print(f'  信心分數: {conf*100:.1f}%（門檻 {CONFIDENCE_THRESHOLD*100:.0f}%）')
        print('  建議結合問診症狀進一步判斷')

else:
    # ── Stage 2: healthy vs symptomatic ──
    X_s2    = scaler_s2.transform(X_input)
    prob_s2 = clf_s2.predict_proba(X_s2)[0]
    pred_s2 = encoder_s2.classes_[np.argmax(prob_s2)]

    healthy_idx     = list(encoder_s2.classes_).index('healthy')
    symptomatic_idx = list(encoder_s2.classes_).index('symptomatic')

    print('Stage 2: Healthy vs Symptomatic')
    print(f'  healthy:     {prob_s2[healthy_idx]*100:.1f}%')
    print(f'  symptomatic: {prob_s2[symptomatic_idx]*100:.1f}%')
    print(f'  → 判定: {pred_s2}')
    print()
    print('=' * 45)

    if pred_s2 == 'healthy':
        conf = prob_s2[healthy_idx]
        if conf >= CONFIDENCE_THRESHOLD:
            print(f'最終結果: Healthy')
            print(f'  信心分數: {conf*100:.1f}%')
        else:
            print(f'最終結果: 疑似 Healthy（信心不足）')
            print(f'  信心分數: {conf*100:.1f}%（門檻 {CONFIDENCE_THRESHOLD*100:.0f}%）')
            print('  建議結合問診症狀進一步判斷')
    else:
        # symptomatic → 輸出三類信心分數
        p_covid       = prob_s1[covid_idx]
        p_noncovid    = prob_s1[noncovid_idx]
        p_healthy     = p_noncovid * prob_s2[healthy_idx]
        p_symptomatic = p_noncovid * prob_s2[symptomatic_idx]

        total = p_covid + p_healthy + p_symptomatic
        p_covid       /= total
        p_healthy     /= total
        p_symptomatic /= total

        conf = prob_s2[symptomatic_idx]
        if conf >= CONFIDENCE_THRESHOLD:
            print('最終結果: Symptomatic')
        else:
            print('最終結果: Symptomatic（信心不足，建議結合問診）')

        print()
        print('各類別信心分數:')
        for cls, prob in sorted(
            [('covid', p_covid), ('healthy', p_healthy), ('symptomatic', p_symptomatic)],
            key=lambda x: -x[1]
        ):
            bar = '█' * int(prob * 30)
            print(f'  {cls:<15} {prob*100:5.1f}%  {bar}')

print()
print(f'真實標籤: {sample["label"]}')
if pred_s1 == 'covid':
    final_pred = 'covid'
else:
    final_pred = pred_s2
print(f'結果: {"✓ 正確" if final_pred == sample["label"] else "✗ 錯誤"}')

Stage 1: COVID vs Non-COVID
  covid:     42.7%
  non-covid: 57.3%
  → 判定: non-covid

Stage 2: Healthy vs Symptomatic
  healthy:     39.8%
  symptomatic: 60.2%
  → 判定: symptomatic

最終結果: Symptomatic

各類別信心分數:
  covid            42.7%  ████████████
  symptomatic      34.5%  ██████████
  healthy          22.8%  ██████

真實標籤: symptomatic
結果: ✓ 正確
